In [80]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [81]:
from randomness import *
from df_metrics import *
from simulation_exact import *
from utils import *
from data_analysis import *
from main import *
import numpy as np
import warnings
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from joblib import Parallel, delayed
from itertools import product

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)
#seed = 42
#np.random.seed(seed)
#rng = np.random.default_rng(seed)

In [ ]:
def generate_beta_data(n_neg, n_pos, rng, d, 
                       alpha_neg, beta_neg, alpha_pos, beta_pos):
    neg_samples = rng.beta(alpha_neg, beta_neg, size=(n_neg, d))
    
    # Sample positive points
    pos_samples = rng.beta(alpha_pos, beta_pos, size=(n_pos, d))
    
    # Combine and return
    samples = np.vstack([neg_samples, pos_samples])

    y = np.concatenate([
        -1 * np.ones(n[1]),
        np.ones(n[0])
    ])

    v = np.ones(n[0]+n[1])

    return X, y, v

In [ ]:
def generate_data_d(n, mu1, mu2, rng, sigma1, sigma2 = 1.0, d=1):
    X1 = rng.normal(loc=mu1, scale=sigma1, size=(n[1], d))
    X2 = rng.normal(loc=mu2, scale=sigma2, size=(n[0], d))
    X = np.vstack([X1, X2])
    y = np.concatenate([
        -1 * np.ones(n[1]),
        np.ones(n[0])
    ])

    v = np.ones(n[0]+n[1])

    return X, y, v

In [ ]:
# 1. choose parameters 
n_pos = 50 # number of data points in a sample with label 1
n_neg = 50 # number of data points in a sample with label -1
n = [n_pos, n_neg]
n_val = [int(n_pos * 2), int(n_neg * 2)]
mu_pos = 0.5 # class 1 mean
mu_neg = -0.5 # class -1 mean
mu = [mu_pos, mu_neg]
# ---------------------------------
use_loss = 'hinge' #'log' or 'hinge' or 'squared_hinge' 
sigma_loss = 1.0 
c = 1.0 # choose c without scaling 
show_plots = False # set to True to see plots of the binary search
is_throw = True # set to True to throw out points outside margin --> only when loss is lipschitz. 
#TODO: add is throw to outside_inside function 
k = None # if none will be set based on data max norm
k_coef = 1.0 # coefficient to multiply k with
fit_intercept = True # whether to fit intercept in SVM model
# ---------------------------------
ds = [2, 4, 8, 16, 32, 64] # dimensions to run experiments on
T = 50  # number of trials
sigmas = np.linspace(1.5, 0.1, 31) # different sigmas to run the experiments on


In [ ]:
# 3. run exact sisigmalation
# returns df with cols "agent" | "true_v" | "critical_v" | "allocation" | "welfare" | "utility"
# utility = welfare - payment
# welfare = allocation * true_v
# critical_v = payment 

def compute_metrics_t(t):
    dfs = []
    for d in ds:
        for sigma in sigmas:
            seed = hash((t, d, sigma)) % (2**32)
            rng = np.random.default_rng(seed)
            x, y, v = generate_data_d(n, mu_pos, mu_neg, rng, sigma1 = sigma, d = d)
            x_val, y_val, _ = generate_data_d(n_val, mu_pos, mu_neg, rng, sigma1 = sigma, d = d)
            
            df_exact, svm_model = run_exact(x, y, v, c, use_loss, sigma_loss=sigma_loss, plot=show_plots, 
                                    is_throw=is_throw, k=k, k_coef=k_coef, fit_intercept=fit_intercept)
            df_exact['t'] = t
            df_exact['d'] = d
            df_exact['sigma'] = sigma
            df_exact['valid_acc'] = svm_model.score(x_val,y_val)
            df_exact['label'] = y[df_exact['agent'].astype(int)]
            dfs.append(df_exact)

    return pd.concat(dfs, ignore_index=True)

In [ ]:
mega_df = pd.concat( 
    Parallel(n_jobs=-1, backend="loky")(
        delayed(compute_metrics_t)(t) for t in range(T)
        ), 
        ignore_index=True
)

In [ ]:
plot_mean_num_payers(mega_df, labels = [1, -1])
plot_mean_payment(mega_df, labels = [1, -1])
plot_mean_payment_sd(mega_df, labels = [1, -1])

#welfare metrics plots
plot_mean_welfare(mega_df, labels = [1, -1])
plot_mean_welfare_sd(mega_df, labels = [1, -1])

# utility metrics plots
plot_mean_utility(mega_df, labels = [1, -1])
plot_mean_utility_sd(mega_df, labels = [1, -1])


# accuracy metrics plots
plot_mean_accuracy_train(mega_df, labels = [1, -1])

plot_mean_accuracy_validation(mega_df, labels = [1, -1])

# percent of points in [0, beta] interval 
plot_percent_relevant(mega_df, labels = [1, -1])

TODO: <br>
<ul>
     <li>create micro notebook </li>
     <li>check for beta data </li>
</ul>

TODO: different loss functions
TODO: lambda , over and underfitting scatter plot where each point is lambda x-axis is acc and y is num\sum of payments , lambda *
TODO: kernels: use more expressive , try with data that needs kernels 
TODO: play with v 
TODO: mean and sd of random payment calculation algorithm 

